# Total Energy + Post Processing

Calculate the total energy of a material with optional post-processing (PP) using a DFT workflow on the Mat3ra platform.

<h2 style="color:green">Usage</h2>

1. Set material and calculation parameters in cells 1.2 and 1.3 below (or use the default values).
1. Click "Run" > "Run All" to run all cells.
1. Wait for the job to complete.
1. Scroll down to view the result.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure parameters for material, workflow, post-processing property, compute resources, and job.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Create material: materials are read from the `../uploads` folder — place files there manually or run a material creation notebook first. If the material is not found by name, Standata is used as a fallback. The material is then saved to the platform.
1. Create workflow and set its parameters: select application, load the PP subworkflow from Standata (or Total Energy if no PP), adjust PP-specific parameters, optionally add relaxation, and save the workflow to the platform.
1. Configure compute: get list of clusters and create compute configuration with selected cluster, queue, and number of processors.
1. Create the job with material and workflow configuration: assemble the job from material, workflow, project, and compute configuration.
1. Submit the job and monitor the status: submit the job and wait for completion.
1. Retrieve results: get and display total energy and the selected post-processing property.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
import os
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
MATERIAL_NAME = "Silicon"

# 4. Workflow parameters
APPLICATION_NAME = "espresso"
ADD_RELAXATION = False

# Post-processing (PP) properties to calculate — list one or more, or use [] for total energy only:
POST_PROCESSING_PROPERTIES = ["wavefunction_amplitude"] # e.g., ["wavefunction_amplitude", "average_electrostatic_potential"]

# Model parameters
MODEL_SUBTYPE = "gga"  # or "lda"

MY_WORKFLOW_NAME = "Total Energy" + (
    " + " + ", ".join(
        p.replace("_", " ").title() for p in POST_PROCESSING_PROPERTIES) if POST_PROCESSING_PROPERTIES else "")

# 5. Compute parameters
CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds

### 1.3. Set specific post-processing parameters

In [ ]:
# Method parameters
PSEUDOPOTENTIAL_TYPE = "us"  # ultrasoft pseudopotentials; alternatives: "nc" (norm-conserving), "paw"
FUNCTIONAL = "pbe"  # for gga: pbe, pbesol, etc.; for lda: pz, etc.

# set k-grid for relaxation and SCF (if not set, KPPRA is used by default)
RELAXATION_KGRID = None  # e.g., [4, 4, 4]
SCF_KGRID = None  # e.g., [4, 4, 4]

# Wavefunction amplitude parameters
# Band selection: "VBM" (Valence Band Maximum), "CBM" (Conduction Band Minimum = VBM+1), or an integer (1-based custom band index)
WFN_BAND = "VBM"
# XY fractional (crystal) coordinates of the sampling point; plot is always 1D along Z
WFN_X = 0.0  # x0(1) in fractional crystal coordinates [0, 1)
WFN_Y = 0.0  # x0(2) in fractional crystal coordinates [0, 1)

# Average electrostatic potential parameters
AVG_SAMPLING_SCALE = 1.0  # macroscopic average period factor
AVG_NUM_POINTS = 3000  # number of points along the averaging direction
AVG_SMOOTHING_WINDOW = 3.0  # smoothing window length (in Bohr)

## 2. Authenticate and initialize API client
### 2.1. Authenticate

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Create material
### 3.1. Load material from local file (or Standata)

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize
from mat3ra.notebooks_utils.material import load_material_from_folder

material = load_material_from_folder(FOLDER, MATERIAL_NAME) or Material.create(
    Materials.get_by_name_first_match(MATERIAL_NAME))

visualize(material)

### 3.2. Save material to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_material_response = get_or_create_material(client, material, ACCOUNT_ID)
saved_material = Material.create(saved_material_response)

## 4. Configure workflow
### 4.1. Select application

In [ ]:
from mat3ra.standata.applications import ApplicationStandata
from mat3ra.ade.application import Application

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")

### 4.2. Load base workflow and find target subworkflow

Each PP property maps to a dedicated standata subworkflow that already includes the SCF step.

In [ ]:
from mat3ra.standata.subworkflows import SubworkflowStandata
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.wode.subworkflows import Subworkflow
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow

PP_SUBWORKFLOW_NAME_MAP = {
    "wavefunction_amplitude": "wavefunction_amplitude.json",
    "average_electrostatic_potential": "average_electrostatic_potential.json",
}


def load_subworkflow(filename):
    swf = Subworkflow(**SubworkflowStandata.filter_by_application(app.name).get_by_name_first_match(filename))
    swf.application = app
    return swf


if POST_PROCESSING_PROPERTIES:
    # Load first PP subworkflow as base (includes pw_scf).
    # Append non-SCF units from each additional PP so SCF runs only once.
    base_subworkflow = load_subworkflow(PP_SUBWORKFLOW_NAME_MAP[POST_PROCESSING_PROPERTIES[0]])
    for pp_name in POST_PROCESSING_PROPERTIES[1:]:
        pp_swf = load_subworkflow(PP_SUBWORKFLOW_NAME_MAP[pp_name])
        for unit in pp_swf.units:
            if unit.name != "pw_scf":
                base_subworkflow.units.append(unit)
    print(f"✅ Subworkflow with {POST_PROCESSING_PROPERTIES} ({len(base_subworkflow.units)} units)")
else:
    te_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match("total_energy.json")
    base_subworkflow = Subworkflow(**te_config["subworkflows"][0])
    print("✅ Using Total Energy subworkflow (no PP)")

workflow = Workflow(name=MY_WORKFLOW_NAME)
workflow.add_subworkflow(base_subworkflow)
visualize_workflow(workflow)

### 4.3. Apply post-processing parameters to workflow
#### 4.3.1. Wavefunction amplitude: band selection and plot plane

In [ ]:
if "wavefunction_amplitude" in POST_PROCESSING_PROPERTIES:
    swf = workflow.subworkflows[0]

    select_band_unit = swf.get_unit_by_name(name="Select Band")
    if select_band_unit is not None:
        if WFN_BAND == "VBM":
            select_band_unit.value = "KBAND_VALUE_BELOW_EF"
        elif WFN_BAND == "CBM":
            select_band_unit.value = "KBAND_VALUE_BELOW_EF + 1"
        else:
            select_band_unit.value = str(int(WFN_BAND))
        swf.set_unit(select_band_unit)
        print(f"✅ Band selection set to: {WFN_BAND} (value: {select_band_unit.value})")

    pp_wfn_unit = swf.get_unit_by_name(name="pp_wfn")
    if pp_wfn_unit is not None:
        for inp in pp_wfn_unit.input:
            inp["rendered"] = inp["rendered"].replace(
                "x0(1) = 0.0, x0(2) = 0.0", f"x0(1) = {WFN_X}, x0(2) = {WFN_Y}"
            )
        swf.set_unit(pp_wfn_unit)
        print(f"✅ Sampling point set to: x={WFN_X}, y={WFN_Y} (crystal coordinates)")

#### 4.3.2. Average electrostatic potential: sampling scale, number of points, smoothing window

In [ ]:
if "average_electrostatic_potential" in POST_PROCESSING_PROPERTIES:
    swf = workflow.subworkflows[0]
    avg_unit = swf.get_unit_by_name(name="average ESP")
    if avg_unit is not None:
        new_average_in = f"1\npp.dat\n{AVG_SAMPLING_SCALE}\n{AVG_NUM_POINTS}\n3\n{AVG_SMOOTHING_WINDOW}\n"
        for inp in avg_unit.input:
            inp["rendered"] = new_average_in
        swf.set_unit(avg_unit)
        print(
            f"✅ average.in updated: scale={AVG_SAMPLING_SCALE}, "
            f"npoints={AVG_NUM_POINTS}, window={AVG_SMOOTHING_WINDOW}"
        )

### 4.4. Add relaxation (optional)

In [ ]:
if ADD_RELAXATION:
    workflow.add_relaxation()
    visualize_workflow(workflow)

### 4.5. Set Model and its parameters (physics)

In [ ]:
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.mode.model import Model

model_config = ModelTreeStandata.get_model_by_parameters(
    type="dft", subtype=MODEL_SUBTYPE, functional=FUNCTIONAL
)
model_config["method"] = {"type": "pseudopotential", "subtype": PSEUDOPOTENTIAL_TYPE}
model = Model.create(model_config)

for subworkflow in workflow.subworkflows:
    subworkflow.model = model

visualize_workflow(workflow)

### 4.6. Modify Method (computational parameters): k-grid for relaxation and SCF

In [ ]:
from mat3ra.wode.context.providers import PointsGridDataProvider

pp_subworkflow = workflow.subworkflows[1 if ADD_RELAXATION else 0]

if RELAXATION_KGRID is not None and ADD_RELAXATION:
    unit = workflow.subworkflows[0].get_unit_by_name(name_regex="relax")
    unit.add_context(PointsGridDataProvider(dimensions=RELAXATION_KGRID, isEdited=True).yield_data())
    workflow.subworkflows[0].set_unit(unit)

if SCF_KGRID is not None:
    unit = pp_subworkflow.get_unit_by_name(name="pw_scf")
    unit.add_context(PointsGridDataProvider(dimensions=SCF_KGRID, isEdited=True).yield_data())
    pp_subworkflow.set_unit(unit)

visualize_workflow(workflow)

### 4.7. Preview final workflow

In [ ]:
visualize_workflow(workflow)
workflow.to_dict()

### 4.8. Save workflow to collection

In [ ]:
from utils.generic import dict_to_namespace
from mat3ra.notebooks_utils.core.entity.workflow.api import get_or_create_workflow

saved_workflow_response = get_or_create_workflow(client, workflow, ACCOUNT_ID)
saved_workflow = Workflow.create(saved_workflow_response)
print(f"Workflow ID: {saved_workflow.id}")

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create compute configuration

In [ ]:
from mat3ra.ide.compute import Compute

# Select cluster: use specified name if provided, otherwise use first available
if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(
    cluster=cluster,
    queue=QUEUE_NAME,
    ppn=PPN
)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

## 6. Create the job
### 6.1. Create job

In [ ]:
from mat3ra.notebooks_utils.api.job import create_job
from mat3ra.notebooks_utils.ui import display_JSON

print(f"Material: {saved_material.id}")
print(f"Workflow: {saved_workflow.id}")
print(f"Project: {project_id}")

job_name = MY_WORKFLOW_NAME + " " + saved_material.formula + " " + timestamp
job_response = create_job(
    api_client=client,
    materials=[saved_material],
    workflow=workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=job_name,
    compute=compute.to_dict(),
)

job = dict_to_namespace(job_response)
job_id = job._id
print("✅ Job created successfully!")
print(f"Job ID: {job_id}")
display_JSON(job_response)

## 7. Submit the job and monitor the status

In [ ]:
client.jobs.submit(job_id)
print(f"✅ Job {job_id} submitted successfully!")

In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async

await wait_for_jobs_to_finish_async(client.jobs, [job_id], poll_interval=POLL_INTERVAL)

## 8. Retrieve and visualize results
### 8.1. Total Energy

In [ ]:
from mat3ra.prode import PropertyName
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

total_energy_data = client.properties.get_for_job(job_id, property_name=PropertyName.scalar.total_energy.value)
visualize_properties(total_energy_data, title="Total Energy")

### 8.2. Wavefunction Amplitude

In [ ]:
if "wavefunction_amplitude" in POST_PROCESSING_PROPERTIES:
    wfn_data = client.properties.get_for_job(
        job_id, property_name=PropertyName.non_scalar.wavefunction_amplitude.value
    )
    visualize_properties(wfn_data, title=f"Wavefunction Amplitude ({WFN_BAND}, x={WFN_X}, y={WFN_Y})")

### 8.3. Average Electrostatic Potential

In [ ]:
if "average_electrostatic_potential" in POST_PROCESSING_PROPERTIES:
    avg_esp_data = client.properties.get_for_job(
        job_id, property_name=PropertyName.non_scalar.average_potential_profile.value
    )
    visualize_properties(
        avg_esp_data,
        title=f"Average Electrostatic Potential (scale={AVG_SAMPLING_SCALE}, npts={AVG_NUM_POINTS}, window={AVG_SMOOTHING_WINDOW})"
    )